# Survival Analysis on the MIMIC Dataset

We consider a longitudinal clinical dataset extracted from the MIMIC database.  
Each patient is associated with several chronological medical reports collected during the follow-up period.

The dataframe contains the following variables:

- `'SUBJECT_ID'`: Unique identifier of a patient. A single patient may have several associated observations (clinical reports).
- `'note_time'`: Datetime $t_j$ corresponding to the $j$-th clinical report of a patient.
- `'embeddings'`:  Sentence embedding extracted from the clinical report. These embeddings are obtained using a biomedical NLP model followed by a SIF aggregation procedure applied to contextual word embeddings. Each embedding is represented as a vector in
  $\mathbb{R}^p$, where typically $p = 768$.
- `'delta_i'`: Event indicator associated with patient $i$. It's formally written as:
  $$
  \delta_i =
  \begin{cases}
  1 & \text{if the event is observed (death)}, \\
  0 & \text{if the patient is censored.}
  \end{cases}
  $$

- `'survival_time'` (or $T_i$): Observed survival time of patient $i$, expressed in days.
- `'time_since_first_note'`: Relative time elapsed since the first available report of the patient.
- `'death_time'`: Datetime of death if the event occurred. This variable is equal to `NaN` for censored patients.

The objective is to transform the sequential trajectory of embeddings into meaningful covariates for survival analysis using the SigBERT pipeline:
1. temporal normalization,
2. path construction,
3. signature feature extraction,
4. Cox proportional hazards modeling.

In [1]:
import sys
print("Jupyter is running on:", sys.executable)

Jupyter is running on: C:\Users\vivia\anaconda3\envs\sigbert\python.exe


In [2]:
import os
import sys
#import skglm
# Add the src directory to the Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
src_path = os.path.abspath(os.path.join(notebook_dir, '.', 'src')) # ADAPT HERE
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Now import our custom modules
from _utils import *
from compression_pkg import *
from survival_analysis_sigbert import *
from metrics_plot_results_pkg import *

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

## I) Data Importation

In [4]:
df_OG = pd.read_csv("C:/Users/vivia/sigbert_mimic/data/part_01_sub/arora_results_part_1_sub_1.csv")
# You can import each part1_sub{j}.csv and merge them into a single `df_OG` dataframe. 

In [5]:
df_OG = convert_date_columns(df_OG)

In [6]:
print_dataset_statistics(df_OG, var_id="SUBJECT_ID", var_death="delta_i")

Total number of patients in the dataset: 15
Total number of medical reports: 380
Average number of reports per patient: 25.33
Number of deceased patients: 6
Number of censored patients: 9


In [7]:
#plot_report_distribution_per_patient(df_OG, var_id="SUBJECT_ID", export_path=None)

## II) Survival Modeling

### II)1) Landmark modeling

In [ ]:
k_comp = 25; max_reports = 221
L_chosen = 36; w_chosen = 12  # Adapt the landmark point and window size by yourself.

# Build the landmark cohort
df_L, patients_L, df_gamma_L = define_landmark_cohort(
    df_OG,
    landmark_months=L_chosen,
    window_months=w_chosen
)

# Count number of clinical reports in the window
df_counts = (
    df_L.groupby("ID")
        .size()
        .reset_index(name="n_reports")
)
mean_reports = df_counts["n_reports"].mean()
std_reports = df_counts["n_reports"].std(ddof=1)

# Train-test construction
df_train_new_OG, test_groups = make_train_test(df_L)

# PCA compression
_, R_comp = pca_compression(df_train_new_OG, k_comp, verbose=True)

df_all = df_L.copy()

In [ ]:
lambda_list_RO = [0.5, 1, 7]#np.arange(0.1, 12.1, 0.1)

results_summary_RO = []
global_lambda_results_RO = {}

start_global_RO = time.time()

for lambda_l1_CV_RO in tqdm(lambda_list_RO):

    print("\n===================================")
    print(f"Running Reports Only for lambda = {lambda_l1_CV_RO}")
    print("===================================")

    start = time.time()

    (
        df_results_RO,
        cph_RO,
        df_survival_RO,
        w_sk_RO,
        scores_RO,
        X_RO,
        y_train_RO,
        y_cox_RO,
        c_index_train_RO,
        c_index_test_list_RO,
        c_index_test_mean_RO,
        c_index_test_std_RO,
        df_survival_test_list_RO
    ) = global_sigbert_mimic_pipeline(
        df_full=df_all,
        df_train=df_train_new_OG,
        test_sets=test_groups,
        projection_matrix=R_comp,
        lambda_l1=lambda_l1_CV_RO,
        signature_order=2,
        use_levy_area=False,
        print_progress=False,
        use_standard_scaling=False
    )

In [ ]:
# --------------------------------------------------
# Number of non-zero Cox coefficients
# --------------------------------------------------
non_zero_coefficients_RO = int(
    (w_sk_RO != 0).sum()
)

# --------------------------------------------------
# Confidence interval on test C-index
# --------------------------------------------------
lower_bound_RO, upper_bound_RO = (
    jackknife_confidence_interval(
        c_index_test_list_RO
    )
)

# --------------------------------------------------
# Execution time (minutes)
# --------------------------------------------------
execution_time_minutes = (
    time.time() - start
) / 60

# --------------------------------------------------
# Store model results
# --------------------------------------------------
global_lambda_results_RO[lambda_l1_CV_RO] = {
    "model": cph_RO,
    "cox_coefficients": w_sk_RO,
    "c_index_train": c_index_train_RO,
    "c_index_test_mean": c_index_test_mean_RO,
    "c_index_test_std": c_index_test_std_RO,
    "non_zero_coefficients": non_zero_coefficients_RO,
    "execution_time_minutes": execution_time_minutes
}

# --------------------------------------------------
# Append summary results
# --------------------------------------------------
results_summary_RO.append({
    "lambda": lambda_l1_CV_RO,
    "C-index Train": np.round(
        c_index_train_RO,
        3
    ),
    "Mean C-index Test": np.round(
        c_index_test_mean_RO,
        3
    ),
    "C-index Test Std": np.round(
        c_index_test_std_RO,
        3
    ),
    "C-index CI95%": (
        f"[{lower_bound_RO:.4f}, "
        f"{upper_bound_RO:.4f}]"
    ),
    "Non-zero Coefficients": (
        non_zero_coefficients_RO
    ),
    "Execution Time (min)": np.round(
        execution_time_minutes,
        2
    )
})

# --------------------------------------------------
# Global results dataframe
# --------------------------------------------------
df_results_RO = pd.DataFrame(
    results_summary_RO
)

# --------------------------------------------------
# Select best lambda
# --------------------------------------------------
best_row_RO = df_results_RO.loc[
    df_results_RO[
        "Mean C-index Test"
    ].idxmax()
]

best_lambda_RO = best_row_RO["lambda"]

print("\n===================================")
print(
    f"Best lambda (Reports Only): "
    f"{best_lambda_RO}"
)
print("===================================")

print(best_row_RO)